# NLP-06 · Transformer Model Families

**Prerequisites:** DL-07, DL-08, NLP-01, and the tiny-language-model checkpoint  
**Estimated mastery time:** 7–10 hours, including the project assessment  
**Next canonical lesson:** NLP-02 · Sentence Embeddings

GPT, BERT, and T5 are not three unrelated inventions. They reuse Transformer
components but enforce different information flow and train against different
targets.

This lesson asks four questions for every family:

1. Which positions may this representation read?
2. What is hidden or shifted in the input?
3. Which target produces the loss?
4. What output structure matches the task?

Brand names are memory aids. Masks, objectives, tensor shapes, and measured
behavior are the transferable knowledge.


> **Canonical learner route · module NLP-06 of 67**
>
> Required prerequisites: **DL-07, DL-08, NLP-01** · Previous: **DL-08** ·
> Next after mastery: **NLP-02** · Expected first-pass workload:
> **5–8 hours**
>
> **Core path:** intuition, objectives, foundations, runnable implementation,
> failure modes, and assessed exercises. **Extension path:** history, production,
> tradeoffs, and interview material may be revisited after the core pass.
> Do not continue merely because every cell ran. Continue when you can complete
> the independent exercise and teach-back without notes. The canonical route in
> `docs/CURRICULUM_PATH.json` is authoritative when section-local file order and
> prerequisite order differ.


## 1 · Learning outcomes

You will be able to:

- derive causal, bidirectional-with-padding, and cross-attention visibility;
- trace `(B,H,T_query,T_key)` for every attention operation;
- construct next-token, masked-token, classification, and source-to-target targets;
- explain why padding masks and masked-language-model tokens have different jobs;
- prove that later input cannot change an earlier decoder logit;
- prove that visible right context can change an earlier encoder representation;
- prove that padded key IDs cannot change valid encoder outputs;
- distinguish decoder self-attention from encoder–decoder cross-attention;
- explain teacher forcing in both decoder-only and encoder–decoder training;
- compare teacher-forced token accuracy with free-running exact match;
- choose a family—or a simpler model—from information and output requirements;
- avoid presenting synthetic memorization as general language ability.

```mermaid
flowchart TD
    A[What information is legal?] --> B{One sequence or two?}
    B -->|one, predict future| C[Causal decoder]
    B -->|one, understand complete input| D[Bidirectional encoder]
    B -->|source and target separate| E[Encoder-decoder]
    C --> F[Next-token loss]
    D --> G[Masked-token or task head]
    E --> H[Shifted target plus cross-attention]
```


<details>
<summary><strong>Mathematics notation support — open when a formula feels dense</strong></summary>

- $x_i$: item or component number $i$; a subscript is a label, not multiplication.
- $n$: number of examples; $d$: number of features or dimensions.
- $\mathbf x$: vector; $X$: matrix; $X^\top$: transpose (rows and columns swap).
- $\hat y$: an estimate or prediction; a hat marks an estimated quantity.
- $\sum$: add repeated terms; $\prod$: multiply repeated terms.
- $\lVert\mathbf x\rVert$: vector length; $|x|$: distance of one number from zero.
- $\frac{\partial f}{\partial x}$: slope of $f$ as $x$ changes while other inputs
  stay fixed; $\nabla f$: vector containing all parameter slopes.
- $P(A\mid B)$: probability of $A$ after restricting attention to cases where
  $B$ is true.
- $\log$ reverses an exponential and turns products into sums.

Read a formula one operator at a time, write object shapes beside vectors and
matrices, and substitute a tiny numeric example. Review PRE-01 through PRE-04 for
worked explanations of these symbols.
</details>


## Student Lesson Companion · NLP-06 · Transformer Model Families

Use this companion during the **core pass**. Write short answers before reading
the extension material; then correct them in a different colour after the lesson.

### Practical problem before history

1. What concrete task or decision are we trying to improve?
2. Why is it difficult with the data, time, labels, or compute available?
3. What is the simplest previous baseline?
4. Where does that baseline fail?
5. What capability must the new concept add?

Section 2 must supply enough evidence to answer these questions. Historical detail
is extension material unless it explains the problem or design constraint.

### Concept, analogy, and analogy limit

After Section 3, explain the concept in one sentence without unexplained jargon.
Map it to one daily-life analogy **or** one concrete visual example. Then state
one place where the mapping breaks; an analogy is a bridge, not a proof.

### Use / avoid / alternatives

Complete this decision table from Sections 7, 9, and 11:

| Decision | Required answer |
|---|---|
| Use it when | Three realistic conditions where its assumptions and benefits fit |
| Avoid it when | Two conditions where it is misleading, unsafe, or wasteful |
| Prefer instead | At least one simpler baseline and one alternative for a failed assumption |
| Evidence | Metric, diagnostic, or constraint that supports the choice |


## 2 · One comparison to orient yourself

| Family | Self-attention visibility | Typical training signal | Natural output |
|---|---|---|---|
| decoder-only | current and earlier target positions | next token | continuation |
| encoder-only | all real input positions | reconstructed token or task label | contextual states or label |
| encoder–decoder | encoder: full source; decoder: earlier target; cross: full source | next target token | transformed sequence |

Analogies:

- A causal decoder writes while the unwritten page is covered.
- A bidirectional encoder reviews a completed page.
- An encoder–decoder writes a separate output while consulting a source page.

The analogy stops there. The models calculate vectors and optimize losses; they do
not read or understand in the human sense.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

candidates = [Path.cwd(), *Path.cwd().parents]
repository_root = next(
    path for path in candidates if (path / "projects/transformer_families").exists()
)
project_root = repository_root / "projects" / "transformer_families"
sys.path.insert(0, str(project_root / "src"))

from transformer_families.models import (
    DecoderOnlyModel,
    EncoderDecoderModel,
    EncoderOnlyModel,
    FamilyConfig,
)
from transformer_families.training import (
    BOS,
    EOS,
    MASK,
    make_classification_data,
    make_cycle_sequences,
    make_reversal_data,
    run_family_lab,
)

torch.manual_seed(42)
config = FamilyConfig()
print(config)


## 3 · Visibility matrices before model names

Let `True` mean a query may read a key.

For target length 4, causal visibility is:

$$
\begin{bmatrix}
1&0&0&0\\
1&1&0&0\\
1&1&1&0\\
1&1&1&1
\end{bmatrix}
$$

A bidirectional encoder permits all **real** keys. If the last two entries are
padding, their columns must be blocked for every query.

Cross-attention is rectangular. With target length $T_t=3$ and source length
$T_s=5$, scores have shape:

$$
(B,H,T_t,T_s)=(B,H,3,5)
$$

Decoder states provide queries; encoder states provide keys and values.


In [ ]:
target_length, source_length = 4, 5
causal_visibility = torch.tril(torch.ones(target_length, target_length, dtype=torch.bool))
source_is_real = torch.tensor([True, True, True, False, False])
encoder_visibility = source_is_real.repeat(source_length, 1)
cross_visibility = source_is_real.repeat(3, 1)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for axis, matrix, title in zip(
    axes,
    [causal_visibility, encoder_visibility, cross_visibility],
    ["causal decoder", "encoder with padding", "cross-attention"],
):
    axis.imshow(matrix, vmin=0, vmax=1, cmap="gray")
    axis.set_xlabel("key position")
    axis.set_ylabel("query position")
    axis.set_title(title)
plt.tight_layout()
plt.show()


## 4 · One shared attention calculation

All three families can reuse:

$$
Q=Q_{states}W_Q,\quad K=C_{states}W_K,\quad V=C_{states}W_V
$$

$$
A=\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_h}}+M\right)
$$

$$
O=AV
$$

For self-attention, `Q_states` and `C_states` are the same sequence. For
cross-attention they are different. $d_h$ is one head's width, and $M$ blocks
illegal keys before softmax.

The block around attention—not the dot-product formula—determines the family.


In [ ]:
def visible_attention(query_states, context_states, allowed):
    scores = query_states @ context_states.transpose(-2, -1)
    scores = scores / np.sqrt(query_states.shape[-1])
    scores = scores.masked_fill(~allowed, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    return weights @ context_states, weights


query_states = torch.tensor([[[1.0, 0.0], [0.0, 1.0]]])
context_states = torch.tensor([[[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]])
allowed = torch.tensor([[[True, True, False], [True, True, True]]])
retrieved, weights = visible_attention(query_states, context_states, allowed)
print("score/weight shape:", weights.shape, "= (B,T_query,T_key)")
print("weights:\n", weights)
print("retrieved:\n", retrieved)
assert torch.all(weights[~allowed] == 0)


## 5 · Decoder-only: shifted targets and causal behavior

For `[4,5,6,7]`:

```text
decoder input:  [4,5,6]
target:         [5,6,7]
```

At position 1, the model sees input IDs 4 and 5 and predicts 6. Teacher forcing
supplies the correct earlier input tokens for all positions during training. The
causal mask prevents it from reading later inputs, so all position losses can be
calculated in parallel.

$$
L_{causal}=-\frac1{BT}\sum_{b,t}\log p(y_{b,t}\mid x_{b,0:t})
$$

Generation remains sequential because the next input ID is the model's newly
selected output.


In [ ]:
cycle = make_cycle_sequences(count=3, length=6)
decoder_inputs, decoder_targets = cycle[:, :-1], cycle[:, 1:]
print("full row: ", cycle[0].tolist())
print("input:    ", decoder_inputs[0].tolist())
print("target:   ", decoder_targets[0].tolist())
assert torch.equal(decoder_inputs[:, 1:], decoder_targets[:, :-1])

decoder = DecoderOnlyModel(config).eval()
original = torch.tensor([[4, 5, 6, 7, 8]])
changed_future = torch.tensor([[4, 5, 6, 12, 13]])
real = torch.ones_like(original, dtype=torch.bool)
with torch.no_grad():
    original_logits = decoder(original, real)
    changed_logits = decoder(changed_future, real)
earlier_difference = (original_logits[:, :3] - changed_logits[:, :3]).abs().max().item()
print("largest earlier-logit change after future perturbation:", earlier_difference)
assert earlier_difference == 0.0


## 6 · Encoder-only: reconstruct or classify with full context

### Masked-token prediction

A dedicated `[MASK]` ID replaces selected input tokens. The targets retain the
originals, and loss is calculated only at selected positions $R$:

$$
L_{MLM}=-\frac1{|R|}\sum_{t\in R}\log p(x_t\mid \widetilde{x})
$$

`[MASK]` is content corruption. A **padding mask** is an attention rule that keeps
nonexistent batch filler from becoming a key. They are not interchangeable.

### Classification

A sequence representation can come from a special classification token or masked
mean pooling. The project uses:

$$
h_{pool}=\frac{\sum_t m_th_t}{\max(1,\sum_t m_t)}
$$

where $m_t=1$ for real tokens and 0 for padding. A linear head maps $h_{pool}$ to
class logits.


In [ ]:
original_tokens = make_cycle_sequences(count=2, length=7)
masked_inputs = original_tokens.clone()
mask_position = 3
masked_targets = original_tokens[:, mask_position]
masked_inputs[:, mask_position] = MASK
print("original:", original_tokens[0].tolist())
print("corrupted:", masked_inputs[0].tolist())
print("target at hidden position:", int(masked_targets[0]))

encoder = EncoderOnlyModel(config).eval()
first = torch.tensor([[4, 5, 6, 7, 8]])
changed_right_context = torch.tensor([[4, 5, 6, 7, 12]])
all_real = torch.ones_like(first, dtype=torch.bool)
with torch.no_grad():
    first_hidden = encoder.encode(first, all_real)
    changed_hidden = encoder.encode(changed_right_context, all_real)
right_context_effect = (first_hidden[:, 0] - changed_hidden[:, 0]).abs().max().item()
print("change at position 0 after changing visible position 4:", right_context_effect)
assert right_context_effect > 1e-6


In [ ]:
# Padding IDs themselves may vary; valid outputs must not care because those keys
# are masked. Padded query rows may still exist, so compare only real positions.
padded_a = torch.tensor([[4, 5, 6, 0, 0]])
padded_b = torch.tensor([[4, 5, 6, 12, 13]])
padding_mask = torch.tensor([[True, True, True, False, False]])
with torch.no_grad():
    hidden_a = encoder.encode(padded_a, padding_mask)
    hidden_b = encoder.encode(padded_b, padding_mask)
valid_difference = (hidden_a[:, :3] - hidden_b[:, :3]).abs().max().item()
print("valid-output change after altering padded IDs:", valid_difference)
assert valid_difference == 0.0

class_inputs, class_targets = make_classification_data(count=4)
class_mask = torch.ones_like(class_inputs, dtype=torch.bool)
print("classification logits shape:", encoder.classify(class_inputs, class_mask).shape)
print("class targets:", class_targets.tolist())


## 7 · Encoder–decoder: source states plus shifted target

Three attention operations coexist:

| Operation | Queries | Keys/values | Mask |
|---|---|---|---|
| encoder self-attention | source | source | source padding |
| decoder self-attention | target prefix | target prefix | causal + target padding |
| decoder cross-attention | decoder states | encoder states | source padding |

For source `[4,6,8]` and desired target `[8,6,4,EOS]`:

```text
decoder input: [BOS,8,6,4]
target:        [8,6,4,EOS]
```

This is teacher forcing again. During free-running generation, the decoder receives
its own earlier outputs instead.


In [ ]:
source, target_inputs, target_outputs = make_reversal_data(count=3, source_length=5)
print("source:       ", source[0].tolist())
print("decoder input:", target_inputs[0].tolist(), "starts with BOS", BOS)
print("target:       ", target_outputs[0].tolist(), "ends with EOS", EOS)
assert target_inputs[0, 0] == BOS
assert target_outputs[0, -1] == EOS
assert torch.equal(target_inputs[:, 1:], target_outputs[:, :-1])

encoder_decoder = EncoderDecoderModel(config).eval()
source_mask = torch.ones_like(source, dtype=torch.bool)
target_mask = torch.ones_like(target_inputs, dtype=torch.bool)
with torch.no_grad():
    base_logits = encoder_decoder(source, target_inputs, source_mask, target_mask)
    changed_source = source.clone()
    changed_source[:, -1] = 13
    source_changed_logits = encoder_decoder(
        changed_source, target_inputs, source_mask, target_mask
    )
    changed_target_future = target_inputs.clone()
    changed_target_future[:, 3:] = 12
    target_changed_logits = encoder_decoder(
        source, changed_target_future, source_mask, target_mask
    )

print("source change affects first decoder logit:",
      (base_logits[:, 0] - source_changed_logits[:, 0]).abs().max().item())
print("future target change affects first three logits:",
      (base_logits[:, :3] - target_changed_logits[:, :3]).abs().max().item())
print("cross-attention score shape:",
      encoder_decoder.decoder_blocks[0].cross_attention.last_score_shape)
assert (base_logits[:, 0] - source_changed_logits[:, 0]).abs().max() > 1e-6
assert torch.allclose(base_logits[:, :3], target_changed_logits[:, :3], atol=1e-6)


## 8 · Controlled training lab

The local project trains four objectives with one shared configuration:

- decoder next-token prediction on digit cycles;
- encoder masked-token reconstruction;
- encoder sequence classification;
- encoder–decoder sequence reversal.

These are wiring diagnostics. They show that gradients can teach each information
path. They do not estimate performance on natural language or unseen distributions.


In [ ]:
report = run_family_lab(seed=42, steps=120)
evidence_table = pd.DataFrame(
    [
        {
            "objective": "decoder next token",
            "initial loss": report["gpt_decoder_only"]["initial_loss"],
            "final loss": report["gpt_decoder_only"]["final_loss"],
            "accuracy": report["gpt_decoder_only"]["token_accuracy"],
        },
        {
            "objective": "encoder masked token",
            "initial loss": report["bert_encoder_only"]["mlm_initial_loss"],
            "final loss": report["bert_encoder_only"]["mlm_final_loss"],
            "accuracy": report["bert_encoder_only"]["mlm_accuracy"],
        },
        {
            "objective": "encoder classification",
            "initial loss": report["bert_encoder_only"]["classification_initial_loss"],
            "final loss": report["bert_encoder_only"]["classification_final_loss"],
            "accuracy": report["bert_encoder_only"]["classification_accuracy"],
        },
        {
            "objective": "source-to-target reversal",
            "initial loss": report["t5_encoder_decoder"]["initial_loss"],
            "final loss": report["t5_encoder_decoder"]["final_loss"],
            "accuracy": report["t5_encoder_decoder"]["teacher_forced_token_accuracy"],
        },
    ]
)
display(evidence_table)
print("free-running reversal exact match:",
      report["t5_encoder_decoder"]["greedy_exact_match_first_eight"])
assert np.all(evidence_table["final loss"] < evidence_table["initial loss"])


Teacher-forced token accuracy can look stronger than free-running generation because
every decoder step receives the correct earlier target during training evaluation.
Once one generated token is wrong, later inputs change. Report both metrics and do
not label teacher-forced accuracy as generation quality.


## 9 · Choosing a family from the task

| Need | First candidate | Why | Baseline to try |
|---|---|---|---|
| open continuation | decoder-only | causal objective matches output | n-gram or template |
| complete-text classification | encoder-only | both-side context and compact task head | TF-IDF + logistic regression |
| sentence representation | encoder-only, then contrastive adaptation | one contextual state per input token | TF-IDF or averaged embeddings |
| explicit source-to-output transformation | encoder–decoder | source and target flows stay separate | rules or retrieval/template |
| exact document lookup | retrieval | no synthesis required | lexical search |

A decoder-only model can be prompted to perform many tasks. That does not make it
the cheapest, fastest, easiest to evaluate, or most natural architecture for each
task.


## 10 · Common mistakes and diagnostics

| Symptom | Likely cause | Behavioral test | Fix |
|---|---|---|---|
| decoder loss impossibly easy | future leakage | perturb future input | restore causal mask |
| encoder cannot use right context | causal mask copied from decoder | perturb later real token | use bidirectional visibility |
| padding changes valid states | padded keys visible | alter only padded IDs | key-padding mask before softmax |
| MLM loss trains every position | corruption mask confused with loss mask | inspect selected indices | gather loss only at hidden positions |
| seq2seq ignores source | cross-attention absent/miswired | change source, freeze prefix | decoder Q; encoder K/V |
| seq2seq target is off by one | BOS/target shift wrong | print first row | `[BOS,*y[:-1]] → y` |
| teacher-forced score high, generation poor | exposure mismatch | free-running exact match | report both; improve training/data |
| synthetic accuracy called mastery | memorization only | held-out rule variation | narrow claim; add generalization split |

Also mask padded **loss targets**, not only attention keys, when variable-length
outputs are batched. An attention mask controls information; an ignore index controls
which target positions contribute to loss.


## 11 · Production boundaries

Real checkpoints add subword tokenization, special-token conventions, pretrained
weights, architecture-specific normalization and positions, dropout, large-scale
data, and versioned generation settings. Loading a class named “causal LM” or
“masked LM” does not verify its data license, context contract, calibration,
robustness, latency, or fitness for a business decision.

Pin model and tokenizer revisions together. Test mask behavior on the exact library
API. Evaluate on one shared task distribution and include a simpler baseline.
No hosted API or external download is required for this lesson.


## 12 · Student check

1. Which axis differs between target and source in cross-attention scores?
2. Why can BERT use a later token to reconstruct an earlier masked token?
3. Why can a plain bidirectional encoder not generate left-to-right safely?
4. What is the difference between `[MASK]`, a padding mask, and an ignored loss target?
5. Write GPT input and target rows for five IDs.
6. Write T5 decoder input and target rows using BOS and EOS.
7. Where do Q, K, and V come from in decoder cross-attention?
8. Why can teacher-forced and free-running metrics disagree?
9. Why is encoder-only a natural starting point for sentence embeddings?
10. Name a realistic task where no Transformer family is the best first choice.


## 13 · Practice and mastery project

**Beginner**

1. Draw all masks for source length 4 and target length 3.
2. Calculate one masked-token loss when the correct-token probability is `0.8`.

**Intermediate**

3. Mask two encoder positions and calculate loss only at those positions.
4. Intentionally remove decoder causality, predict which behavioral test fails,
   observe it fail, and restore the mask.
5. Add padded source and target rows. Mask attention keys and loss targets separately.

**Challenge**

6. Create held-out digit rules for all four objectives. Compare memorization and
   generalization across three seeds, then recommend the simplest adequate family.

Complete `projects/transformer_families/MASTERY_CHECKPOINT.md`. Passing requires all
automated invariants, real loss reduction for every objective, a repaired broken
mask, at least **17/20**, and no zero on the causal, cross-attention, or target-shift
explanations.


## 14 · Summary and memory aid

The families differ less in the attention equation than in legal information flow,
corruption, targets, and output structure:

- Decoder-only: hide the future and predict the next token.
- Encoder-only: read the complete real input and learn representations or labels.
- Encoder–decoder: encode a source, then causally produce a separate target while
  cross-attending to source states.

**Memory aid:** *Cover the future to continue, uncover the input to represent, and
keep source and target separate to transform.*

NLP-02 comes next because sentence embeddings require a principled way to pool and
train encoder token states—not merely select an encoder-shaped architecture.


## Lesson Close · Summary, Student Check, and Memory Aid

### Five short student checks

1. What practical problem does **NLP-06 · Transformer Model Families** solve?
2. What is its central mechanism in simple language?
3. Which assumption or limitation is easiest to forget?
4. What output or diagnostic tells you it worked as intended?
5. When would you choose a simpler or related alternative?

### Plain-language summary

Complete four sentences without notes: **The problem is… The concept works by…
I would use it when… I would avoid it when…** Compare your answer with the
objectives, failure modes, tradeoff analysis, and teach-back section.

### One-sentence memory aid

**Remember NLP-06 · Transformer Model Families: start from the problem, trace the mechanism, verify the
evidence, and use it only when its assumptions fit.** Replace this general aid
with your own topic-specific sentence of no more than 20 words.

The lesson is complete only after the Required Core Mastery Gate, not after the
final code cell runs.


## Required Core Mastery Gate · NLP-06

Complete this gate before treating the module as finished. The longer exercises
in Section 14 are extension practice unless the module says otherwise.

**Worked example:** rerun the smallest worked example in this notebook. Annotate
every input, output, shape or unit, and the claim the result supports.

**Guided practice (20–30 min):** change one input in that example. Before running
it, predict the direction of change and explain why. Use the nearest preceding
formula or algorithm step as a hint. **Self-check:** compare prediction with the
result and explain any mismatch rather than editing the prediction afterward.

**Independent practice (30–45 min):** on fresh tiny data, reproduce the module's
central operation without copying the completed cell. State assumptions, expected
output, and one invalid input. **Self-check:** verify with an assertion, a manual
calculation, or a trusted library implementation.

**Challenge (30–60 min):** create one failure described in Section 7, detect it
using evidence, and repair it without changing unrelated variables.

**Solution and scoring rubric:** 2 points for a correct prediction, 2 for a
runnable independent implementation, 2 for an objective self-check, 2 for failure
diagnosis, and 2 for teach-back without notes. Common mistakes: copying before
attempting, using later-module concepts as unexplained shortcuts, evaluating on
training data, and continuing because cells ran. **Readiness threshold: 8/10**,
including both independent implementation and teach-back points. If below 8,
revisit the named prerequisite in the route card and retry with different data.
